# Black Friday Sales Prediction

## Objective
- Predict purchase amount
- Analyze spending behavior
- Build regression models


## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

## Load Data

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

## Data Cleaning

In [ ]:
train['Product_Category_2'].fillna(0, inplace=True)
train['Product_Category_3'].fillna(0, inplace=True)

test['Product_Category_2'].fillna(0, inplace=True)
test['Product_Category_3'].fillna(0, inplace=True)

train['Stay_In_Current_City_Years'] = train['Stay_In_Current_City_Years'].replace('4+', 4).astype(int)
test['Stay_In_Current_City_Years'] = test['Stay_In_Current_City_Years'].replace('4+', 4).astype(int)

train['Gender'] = train['Gender'].map({'M':1, 'F':0})
test['Gender'] = test['Gender'].map({'M':1, 'F':0})

train['City_Category'] = train['City_Category'].map({'A':0, 'B':1, 'C':2})
test['City_Category'] = test['City_Category'].map({'A':0, 'B':1, 'C':2})

age_map = {'0-17':8,'18-25':21,'26-35':30,'36-45':40,'46-50':48,'51-55':53,'55+':60}
train['Age'] = train['Age'].map(age_map)
test['Age'] = test['Age'].map(age_map)

## EDA (Storytelling with Plots)

In [ ]:
# Purchase Distribution
plt.figure()
plt.hist(train['Purchase'], bins=50)
plt.title("Purchase Distribution")
plt.show()

# Gender vs Purchase
train.groupby('Gender')['Purchase'].mean().plot(kind='bar', title='Avg Purchase by Gender')
plt.show()

# Age vs Purchase
train.groupby('Age')['Purchase'].mean().plot(kind='line', title='Avg Purchase by Age')
plt.show()

# City Category vs Purchase
train.groupby('City_Category')['Purchase'].mean().plot(kind='bar', title='City vs Purchase')
plt.show()

## Feature Engineering

In [ ]:
user_avg = train.groupby('User_ID')['Purchase'].mean()
train['user_avg_purchase'] = train['User_ID'].map(user_avg)
test['user_avg_purchase'] = test['User_ID'].map(user_avg)

prod_avg = train.groupby('Product_ID')['Purchase'].mean()
train['prod_avg_purchase'] = train['Product_ID'].map(prod_avg)
test['prod_avg_purchase'] = test['Product_ID'].map(prod_avg)

prod_count = train['Product_ID'].value_counts()
train['prod_purchase_count'] = train['Product_ID'].map(prod_count)
test['prod_purchase_count'] = test['Product_ID'].map(prod_count)

train['age_gender'] = train['Age'] * train['Gender']
test['age_gender'] = test['Age'] * test['Gender']

train['city_age'] = train['City_Category'] * train['Age']
test['city_age'] = test['City_Category'] * test['Age']

train.fillna(0, inplace=True)
test.fillna(0, inplace=True)

## Prepare Data

In [ ]:
X = train.drop(['Purchase','User_ID','Product_ID'], axis=1)
y = train['Purchase']

X_test = test.drop(['User_ID','Product_ID'], axis=1)

## Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

## Polynomial Features

In [ ]:
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_scaled)
X_test_poly = poly.transform(X_test_scaled)

## Model Training & Comparison

In [ ]:
# Linear
lr = LinearRegression()
lr.fit(X_scaled, y)
pred_lr = lr.predict(X_scaled)
rmse_lr = np.sqrt(mean_squared_error(y, pred_lr))

# Ridge
ridge = Ridge(alpha=10)
ridge.fit(X_scaled, y)
pred_ridge = ridge.predict(X_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y, pred_ridge))

# Polynomial Ridge
ridge_poly = Ridge(alpha=10)
ridge_poly.fit(X_poly, y)
pred_poly = ridge_poly.predict(X_poly)
rmse_poly = np.sqrt(mean_squared_error(y, pred_poly))

print("Linear RMSE:", rmse_lr)
print("Ridge RMSE:", rmse_ridge)
print("Polynomial RMSE:", rmse_poly)

## Hyperparameter Tuning

In [ ]:
grid = GridSearchCV(Ridge(), {'alpha':[0.1,1,10,50]}, cv=3, scoring='neg_root_mean_squared_error')
grid.fit(X_poly, y)

print("Best Params:", grid.best_params_)
print("Best RMSE:", -grid.best_score_)

## Predictions

In [ ]:
best_model = grid.best_estimator_
predictions = best_model.predict(X_test_poly)

## Conclusion

### Key Insights
- Age 26–35 spends the most
- Male users contribute higher revenue
- Product categories strongly influence purchase

### Model Summary
- Best Model: Polynomial Ridge
- RMSE improved with feature engineering

### Business Impact
- Focus on high-value customer segments
- Promote high-performing product categories